In [1]:
from pathlib import Path
import numpy as np 
import shutil
import os 
import cv2
import pickle 
import dask
import random
import dask.array as da
import skimage
import matplotlib.pyplot as plt
from collections import Counter
from tqdm import tqdm
import pandas as pd
import networkx as nx
import math
import tifffile as tf
from dask.diagnostics import ProgressBar
from skimage.color import rgb2hed, hed2rgb


In [ ]:
def resize_chunk(chunk, factor):
    return cv2.resize(chunk, None, fx=1/factor, fy=1/factor)

In [ ]:
def invert_chunk(chunk):
    return cv2.bitwise_not(chunk)

In [ ]:
def white_balance(chunk, white_patch_average, image_average):
    return (chunk * (white_patch_average/image_average)).clip(0, 255).astype((int)).astype(np.uint8)

In [ ]:
def register_images_roi(img1_original_shape, roi_img1, roi_img2, img2_original, img_type):

    if(img_type[0] == 'FL'):
        roi_img1_processed = cv2.convertScaleAbs(roi_img1,  alpha=(255.0/65535.0))
    else:
        roi_img1_processed = cv2.cvtColor(roi_img1, cv2.COLOR_RGB2GRAY)
    
    if(img_type[1] == 'FL'):
        roi_img2_processed = cv2.convertScaleAbs(roi_img2,  alpha=(255.0/65535.0))
    else:
        roi_img2_processed = cv2.cvtColor(roi_img2, cv2.COLOR_RGB2GRAY)
    orb = cv2.SIFT_create()
    (keypoints1, descriptors1) = orb.detectAndCompute(roi_img1_processed, None)
    (keypoints2, descriptors2) = orb.detectAndCompute(roi_img2_processed, None)

    FLANN_INDEX_KDTREE = 1
    index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
    search_params = dict(checks=50)
    flann = cv2.FlannBasedMatcher(index_params, search_params)
    matches = flann.knnMatch(descriptors1, descriptors2, k=2)

    good_matches = []
    for m, n in matches:
        if m.distance < 0.7 * n.distance:
            good_matches.append(m)

    points1 = np.float32([keypoints1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    points2 = np.float32([keypoints2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)

    homography, mask = cv2.findHomography(points1, points2, cv2.RANSAC, 5.0)

    height, width= img1_original_shape[:2]
    registered_image = skimage.transform.warp(img2_original, np.array(homography), output_shape=(height, width))  #cv2.warpPerspective(img2_original, homography, (width, height))

    return homography, registered_image

In [ ]:
key = "" # sample identifier
registration_base_path = r"" # path to registered images
cell_clustering_base_path = r"" # path to cell phenotyping results
pixel_clustering_base_path = r"" # path to pixel clustering results

In [ ]:
raw_mask = tf.imread(r"") # path to mask generated by GNCNN

In [ ]:
raw_DAPI_for_registration = tf.imread(r"") # path to Cycle 1 DAPI

In [ ]:
raw_HE = tf.imread(r"") # path to H&E
raw_PAS = tf.imread(r"") # path to PAS
raw_Trichrome = tf.imread(r"") # path to Trichrome

In [ ]:
factor_downsample = 4

#### Register mask based on PAS registration to H&E

In [ ]:
HE_registered = tf.imread(registration_base_path + '/HE_registered.tif')

In [ ]:
raw_PAS_dask = da.from_array(raw_PAS, chunks=(512, 512, 3))
raw_PAS_rotated  = da.rot90(raw_PAS_dask, k=1) 
white_patch = raw_PAS_rotated[:5000, :5000].compute()
white_patch_average = white_patch[0].mean()
PAS_average = da.mean(raw_PAS_rotated).compute()
PAS_balanced = raw_PAS_rotated.map_blocks(white_balance, white_patch_average, PAS_average)
with ProgressBar():
    PAS_balanced_computed = PAS_balanced.compute()

PAS_resized_computed = cv2.resize(PAS_balanced_computed, (HE_registered.shape[1], HE_registered.shape[0]))

In [ ]:
raw_T_dask = da.from_array(raw_Trichrome, chunks=(512, 512, 3))
raw_T_rotated  = da.rot90(raw_T_dask, k=1) 
white_patch = raw_T_rotated[:5000, :5000].compute()
white_patch_average = white_patch[0].mean()
T_average = da.mean(raw_T_rotated).compute()
T_balanced = raw_T_rotated.map_blocks(white_balance, white_patch_average, T_average)
with ProgressBar():
    T_balanced_computed = T_balanced.compute()

T_resized_computed = cv2.resize(T_balanced_computed, (HE_registered.shape[1], HE_registered.shape[0]))

In [ ]:
HE_region = (HE_registered[:10000, :10000] / np.max(HE_registered[:10000, :10000]) * 255).astype(np.uint8)
PAS_region = (PAS_resized_computed[:10000, :10000] / np.max(PAS_resized_computed[:10000, :10000]) * 255).astype(np.uint8)
T_region = (T_resized_computed[:10000, :10000] / np.max(T_resized_computed[:10000, :10000]) * 255).astype(np.uint8)

In [ ]:
H_PAS, PAS_registered = register_images_roi(HE_registered.shape, HE_region, PAS_region, PAS_resized_computed, ['BF', 'BF'])

In [ ]:
H_T, T_registered = register_images_roi(HE_registered.shape, HE_region, T_region, T_resized_computed, ['BF', 'BF'])

In [ ]:
PAS_registered  = (PAS_registered / np.max(PAS_registered) * 255).astype(np.uint8)

In [ ]:
T_registered = (T_registered / np.max(T_registered) * 255).astype(np.uint8)

In [ ]:
mask_resized = cv2.resize(np.rot90(raw_mask),  (HE_registered.shape[1], HE_registered.shape[0]))
h, w = HE_registered.shape[:2]
mask_registered = skimage.transform.warp(mask_resized, np.array(H_PAS), output_shape=(h, w), order=0)

In [ ]:
mask_registered_uint = (mask_registered/ np.max(mask_registered) * 255).astype(np.uint8)

In [ ]:
contours, _ = cv2.findContours(mask_registered_uint, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

In [ ]:
with open(pixel_clustering_base_path + '/wsi_labeled.pickle', 'rb') as f:
    wsi_labeled = pickle.load(f)
num_clusters = np.nanmax(wsi_labeled)

In [ ]:
output_path = r"" # path to save outputs

In [ ]:
Path(output_path + key).mkdir(exist_ok=True)

In [ ]:
Path(output_path + key + '/PAS_regions').mkdir(exist_ok=True)
Path(output_path + key + '/pixel_cluster_regions').mkdir(exist_ok=True)

In [ ]:
Path(output_path + key + '/pixel_cluster_regions_background_removed').mkdir(exist_ok=True)

In [ ]:
Path(output_path + key + '/contours').mkdir(exist_ok=True)

In [ ]:
Path(output_path + key + '/contours_region').mkdir(exist_ok=True)

In [ ]:
Path(output_path + key + '/HE_regions').mkdir(exist_ok=True)
Path(output_path + key + '/Trichrome_regions').mkdir(exist_ok=True)

In [ ]:
tf.imwrite(output_path + key + '/mask_registered.tif', mask_registered)

In [ ]:
cluster_count_contour_list = []
cluster_frequency_contour_list = []
labels = []

In [ ]:
wsi_labeled_nan_removed = wsi_labeled.copy()
wsi_labeled_nan_removed[np.isnan(wsi_labeled_nan_removed)] = 0

In [ ]:
i = 0
for contour in tqdm(contours):
    mask_contour = np.zeros((mask_registered.shape)) 
    mask_contour = cv2.drawContours(mask_contour, [contour], -1, 255, -1)

    wsi_labeled_contour = wsi_labeled_nan_removed.copy()
    wsi_labeled_contour[mask_contour!=255] = 0
    wsi_labeled_contour = wsi_labeled_contour[wsi_labeled_contour!=False]
    cluster_counter_contour = Counter(wsi_labeled_contour)
    cluster_count_contour = [cluster_counter_contour[i] for i in range(1, int(num_clusters)+1)]
    
    cluster_count_contour_list.append(cluster_count_contour)
    cluster_frequency_contour_list.append([i/sum(cluster_count_contour) for i in cluster_count_contour])

    #export PAS and pixel cluster images 
    x, y, w, h = cv2.boundingRect(contour)
    w_offset = 384-w
    h_offset = 384-h
    
    HE_region  = HE_registered[y-int(np.floor(h_offset/2)):y+h+int(np.ceil(h_offset/2)), x-int(np.floor(w_offset/2)):x+w+int(np.ceil(w_offset/2))]
    PAS_region = PAS_registered[y-int(np.floor(h_offset/2)):y+h+int(np.ceil(h_offset/2)), x-int(np.floor(w_offset/2)):x+w+int(np.ceil(w_offset/2))]
    T_region = T_registered[y-int(np.floor(h_offset/2)):y+h+int(np.ceil(h_offset/2)), x-int(np.floor(w_offset/2)):x+w+int(np.ceil(w_offset/2))]

    tf.imwrite(output_path + key + '/HE_regions/Tile' + str(i+1) + '.tif', HE_region)
    tf.imwrite(output_path + key + '/PAS_regions/Tile' + str(i+1) + '.tif', PAS_region)
    tf.imwrite(output_path + key + '/Trichrome_regions/Tile' + str(i+1) + '.tif', T_region)
    
    wsi_labeled_region = wsi_labeled[y-int(np.floor(h_offset/2)):y+h+int(np.ceil(h_offset/2)), x-int(np.floor(w_offset/2)):x+w+int(np.ceil(w_offset/2))]
    with open(output_path + key + '/pixel_cluster_regions/Tile' + str(i+1) + '.pickle', 'wb') as f: 
        pickle.dump(wsi_labeled_region, f)
    
    mask_contour_region = mask_contour[y-int(np.floor(h_offset/2)):y+h+int(np.ceil(h_offset/2)), x-int(np.floor(w_offset/2)):x+w+int(np.ceil(w_offset/2))]
    wsi_labeled_region_background_removed = wsi_labeled_region.copy()
    wsi_labeled_region_background_removed[mask_contour_region!=255] = 0
    with open(output_path + key + '/pixel_cluster_regions_background_removed/Tile' + str(i+1) + '.pickle', 'wb') as f: 
        pickle.dump(wsi_labeled_region_background_removed, f)

    mask_registered_region_padded = mask_registered[y-int(np.floor(h_offset/2)):y+h+int(np.ceil(h_offset/2)), x-int(np.floor(w_offset/2)):x+w+int(np.ceil(w_offset/2))]
    mask_registered_region = mask_registered[y:y+h, x:x+w]
    tf.imwrite(output_path + key + '/contours_region/Tile' + str(i+1) + '.tif', mask_registered_region_padded)
    tf.imwrite(output_path + key + '/contours/Tile' + str(i+1) + '.tif', mask_registered_region)

    mask_registered_contour = mask_registered.copy()
    mask_registered_contour = mask_registered_contour[mask_contour==255]

    #extract glomerulus classification 
    labels.append(np.argmax(np.bincount((mask_registered_contour))))
        
    i+=1


In [ ]:
value_label_dict = {1: 'Globally Sclerotic Glomerulus', 2: 'Non-Globally Sclerotic Glomerulus'}

In [ ]:
with open(output_path + key + '/cluster_count.pickle', 'wb') as f: 
    pickle.dump(cluster_count_contour_list, f)

In [ ]:
with open(output_path + key + '/cluster_frequency.pickle', 'wb') as f: 
    pickle.dump(cluster_frequency_contour_list, f)

In [ ]:
with open(output_path +  key + '/labels.pickle', 'wb') as f: 
    pickle.dump(labels, f)

#### Create dataset

In [ ]:
keys = [] # sample identifiers

In [ ]:
cluster_count_contour_lists = []

for k in keys: 
    with open(output_path + k + '/cluster_count.pickle', 'rb') as f:
        cluster_count_contour_lists.append(pickle.load(f))


In [6]:
sample_lists = []
for i in range(len(cluster_count_contour_lists)):
    sample_lists.extend([keys[i]]*len(cluster_count_contour_lists[i]))

In [ ]:
with open(output_path + '/sample_label.pickle', 'wb') as f:
    pickle.dump(sample_lists, f)

In [ ]:
cluster_frequency_contour_lists = []

for k in keys: 
    with open(output_path + k + '/cluster_frequency.pickle', 'rb') as f:
        cluster_frequency_contour_lists.append(pickle.load(f))

In [ ]:
labels_lists = []

for k in keys: 
    with open(output_path + k + '/labels.pickle', 'rb') as f:
        labels_lists.append(pickle.load(f))

In [ ]:
cluster_counts = cluster_count_contour_lists[0].copy()
cluster_counts.extend(cluster_count_contour_lists[1])
cluster_counts.extend(cluster_count_contour_lists[2])

In [ ]:
cluster_frequency = cluster_frequency_contour_lists[0].copy()
cluster_frequency.extend(cluster_frequency_contour_lists[1])
cluster_frequency.extend(cluster_frequency_contour_lists[2])

In [ ]:
glomerulus_labels = labels_lists[0].copy()
glomerulus_labels.extend(labels_lists[1])
glomerulus_labels.extend(labels_lists[2])

In [ ]:
for i in range(len(glomerulus_labels)): 
    if(glomerulus_labels[i] == 2):
        glomerulus_labels[i] = 0

In [ ]:
data_count_df = pd.DataFrame(cluster_counts, columns=range(1, int(16)+1))
data_count_df['label'] = glomerulus_labels 

In [ ]:
data_frequency_df = pd.DataFrame(cluster_frequency, columns=range(1, int(16)+1))
data_frequency_df.columns = data_frequency_df.columns.astype(str)
data_frequency_df['label'] = glomerulus_labels 

In [ ]:
data_frequency_df.to_csv(output_path + '/data_frequency_df.csv')